In [44]:
"""
LinNeo — Extracción de Nodos Taxonómicos
=========================================
Lee gbif_taxonomy.csv y genera un CSV por rango taxonómico:
    kingdom_nodes.csv, phylum_nodes.csv, class_nodes.csv,
    order_nodes.csv, family_nodes.csv, genus_nodes.csv

También genera cypher_import_taxonomy.cypher con el código
LOAD CSV listo para pegar en Neo4j Browser.

Uso:
    python extract_taxonomy_nodes.py
"""

'\nLinNeo — Extracción de Nodos Taxonómicos\n=========================================\nLee gbif_taxonomy.csv y genera un CSV por rango taxonómico:\n    kingdom_nodes.csv, phylum_nodes.csv, class_nodes.csv,\n    order_nodes.csv, family_nodes.csv, genus_nodes.csv\n\nTambién genera cypher_import_taxonomy.cypher con el código\nLOAD CSV listo para pegar en Neo4j Browser.\n\nUso:\n    python extract_taxonomy_nodes.py\n'

In [45]:
import pandas as pd
from pathlib import Path
import shutil

In [46]:
BASE_DIR     = Path(r"C:\Users\hecto\OneDrive\Escritorio\Personal\iroFactory\35.LinNeo")
DATA_DIR     = BASE_DIR / "biodiversity_data"
OUTPUT_DIR   = DATA_DIR / "taxonomy_nodes_by_rank"
TAXONOMY_CSV = DATA_DIR / "gbif_taxonomy.csv"
CYPHER_OUT   = BASE_DIR / "cypher_import_taxonomy.cypher"

NEO4J_IMPORT = Path(
    r"C:\Users\hecto\.Neo4jDesktop2\Data\dbmss"
    r"\dbms-d1141410-4b1c-481c-b30f-20f8f80f097b\import"
)

In [47]:
# Cada rango: qué columnas usar de gbif_taxonomy.csv
# key_col  → columna con el ID numérico de GBIF
# name_col → columna con el nombre del taxón
RANKS = [
    {"rank": "Kingdom", "key_col": "kingdom_key", "name_col": "kingdom"},
    {"rank": "Phylum",  "key_col": "phylum_key",  "name_col": "phylum"},
    {"rank": "Class",   "key_col": "class_key",   "name_col": "class"},
    {"rank": "Order",   "key_col": "order_key",   "name_col": "order"},
    {"rank": "Family",  "key_col": "family_key",  "name_col": "family"},
    {"rank": "Genus",   "key_col": "genus_key",   "name_col": "genus"},
]

## 1. Data load

In [48]:
def load_taxonomy(path: Path) -> pd.DataFrame:
    print(f"Cargando {path.name} ...")
    df = pd.read_csv(path, dtype=str, low_memory=False)
    print(f"  {len(df):>10,} filas cargadas")
    print(f"  Columnas: {list(df.columns)}")
    print()
    return df

## 2. Node extraction

In [49]:
def extract_rank_nodes(df: pd.DataFrame, rank_def: dict) -> pd.DataFrame:
    """
    Extrae nodos unicos para un rango dado.
    Resultado: DataFrame con columnas [key, name]
        - key  -> ID numerico del GBIF (entero)
        - name -> nombre del taxon
    Filtra filas sin key, sin nombre, y key == 0 (desconocido en GBIF).
    """
    rank     = rank_def["rank"]
    key_col  = rank_def["key_col"]
    name_col = rank_def["name_col"]

    missing = [c for c in [key_col, name_col] if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{rank}] Columnas no encontradas: {missing}\n"
            f"Columnas disponibles: {list(df.columns)}"
        )

    nodes = (
        df[[key_col, name_col]]
        .rename(columns={key_col: "key", name_col: "name"})
        .dropna(subset=["key", "name"])
        .query("key != '0' and key != '' and name != ''")
        .drop_duplicates(subset=["key"])
        .assign(key=lambda x: x["key"].astype(int))
        .sort_values("key")
        .reset_index(drop=True)
    )

    return nodes

In [50]:
def extract_all_ranks(df: pd.DataFrame, ranks: list, output_dir: Path) -> dict:
    """
    Extrae y guarda los CSVs de todos los rangos Kingdom a Genus.
    Retorna dict: rank -> Path del CSV generado.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    node_files = {}

    print("Extrayendo nodos por rango:")
    print(f"  {'Rango':<10} {'Nodos unicos':>14}   Archivo")
    print(f"  {'---'*10} {'---'*5}   {'---'*10}")

    for r in ranks:
        rank     = r["rank"]
        nodes    = extract_rank_nodes(df, r)
        out_file = output_dir / f"{rank.lower()}_nodes.csv"
        nodes.to_csv(out_file, index=False)
        node_files[rank] = out_file
        print(f"  {rank:<10} {len(nodes):>14,}   {out_file.name}")

    print()
    return node_files

## 3. Species nodes

In [51]:
def extract_species_nodes(df: pd.DataFrame, output_dir: Path) -> Path:
    """
    Extrae nodos de especies desde gbif_taxonomy.csv.

    Filtra:
        - rank == 'species'
        - taxonomic_status == 'accepted'
        - species_key no nulo ni vacio

    Columnas del CSV resultante:
        - key             -> species_key (entero)
        - name            -> canonical_name
        - scientific_name -> scientific_name completo
        - kingdom         -> reino al que pertenece
        - kingdom_key     -> key del reino
    """
    print("Extrayendo nodos de Species ...")

    required_cols = ["species_key", "rank", "taxonomic_status", "canonical_name",
                     "scientific_name", "kingdom", "kingdom_key"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Columnas faltantes para Species: {missing}")

    species = (
        df[
            (df["rank"] == "species") &
            (df["taxonomic_status"] == "accepted")
        ]
        .dropna(subset=["species_key", "canonical_name"])
        .query("species_key != '' and canonical_name != ''")
        [["species_key", "canonical_name", "scientific_name", "kingdom", "kingdom_key"]]
        .rename(columns={
            "species_key":    "key",
            "canonical_name": "name",
        })
        .drop_duplicates(subset=["key"])
        .assign(key=lambda x: x["key"].astype(int))
        .assign(kingdom_key=lambda x: pd.to_numeric(x["kingdom_key"], errors="coerce").astype("Int64"))
        .sort_values("key")
        .reset_index(drop=True)
    )

    out_file = output_dir / "species_nodes.csv"
    species.to_csv(out_file, index=False)

    print(f"  Total especies (accepted): {len(species):,}")
    print(f"  Guardado en: {out_file.name}")
    print()
    print("  Distribucion por reino:")
    for reino, count in species["kingdom"].value_counts().items():
        print(f"    {reino:<20} {count:>10,}")
    print()

    return out_file

## 4. CQL generation

In [52]:
def generate_cypher_load_csv(ranks: list, batch_size: int = 10_000) -> str:
    """
    Genera el codigo Cypher completo para importar los nodos taxonomicos
    (Kingdom a Species) a Neo4j con LOAD CSV WITH HEADERS.

    Parametros
    ----------
    ranks : list[dict]
        Lista de rangos Kingdom a Genus (la misma RANKS del script).
    batch_size : int
        Filas por transaccion en CALL { } IN TRANSACTIONS.

    Retorna
    -------
    str
        Codigo Cypher listo para pegar en Neo4j Browser o guardar en .cypher.

    Notas
    -----
    - Cada bloque es independiente y se puede ejecutar por separado.
    - Usa MERGE para ser idempotente (re-ejecutable sin duplicados).
    - La propiedad de clave se llama {rank_lower}_key para coincidir
      con los nombres de GBIF (kingdom_key, phylum_key, etc.).
    - Species tiene su propio constraint y bloque LOAD CSV.
    """
    sep = "// " + "=" * 60

    lines = [
        sep,
        "// LinNeo -- Import nodos taxonomicos (Kingdom a Species)",
        sep,
        "//",
        "// INSTRUCCIONES:",
        "// 1. Copia los CSVs a la carpeta import/ de tu instancia Neo4j",
        "// 2. Asegurate de estar usando la base de datos LinNeo",
        "//       :use LinNeo",
        "// 3. Ejecuta primero el bloque de constraints",
        "// 4. Luego ejecuta cada bloque LOAD CSV por separado",
        "//",
        "",
    ]

    # Constraints: Kingdom a Genus + Species
    lines += [
        "// -- CONSTRAINTS (ejecutar primero, una sola vez) --",
        "",
    ]
    for r in ranks:
        rank = r["rank"]
        prop = f"{rank.lower()}_key"
        lines.append(
            f"CREATE CONSTRAINT {rank.lower()}_key IF NOT EXISTS "
            f"FOR (n:{rank}) REQUIRE n.{prop} IS UNIQUE;"
        )
    # Species constraint separado al final del bloque
    lines.append(
        "CREATE CONSTRAINT species_key IF NOT EXISTS "
        "FOR (n:Species) REQUIRE n.species_key IS UNIQUE;"
    )
    lines += ["", ""]

    # LOAD CSV Kingdom a Genus
    lines.append("// -- LOAD CSV Kingdom a Genus (ejecutar cada bloque por separado) --")
    lines.append("")

    for r in ranks:
        rank     = r["rank"]
        prop_key = f"{rank.lower()}_key"
        filename = f"{rank.lower()}_nodes.csv"

        lines += [
            f"// {rank}",
            f"LOAD CSV WITH HEADERS FROM 'file:///{filename}' AS row",
            f"CALL (row) {{",
            f"  MERGE (n:{rank} {{{prop_key}: toInteger(row.key)}})",
            f"  SET n.name = row.name",
            f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
            "",
        ]

    # LOAD CSV Species
    lines += [
        "// -- LOAD CSV Species --",
        "// Solo especies accepted. Incluye kingdom_key para trazabilidad.",
        "",
        "LOAD CSV WITH HEADERS FROM 'file:///species_nodes.csv' AS row",
        "CALL (row) {",
        "  MERGE (n:Species {species_key: toInteger(row.key)})",
        "  SET n.name            = row.name",
        "  SET n.scientific_name = row.scientific_name",
        "  SET n.kingdom         = row.kingdom",
        "  SET n.kingdom_key     = toInteger(row.kingdom_key)",
        f"}} IN TRANSACTIONS OF {batch_size} ROWS;",
        "",
        "",
    ]

    # Verificacion
    lines += [
        "// -- VERIFICACION --",
        "",
        "MATCH (n)",
        "RETURN labels(n)[0] AS label, count(*) AS total",
        "ORDER BY total DESC;",
    ]

    return "\n".join(lines)

## 5. CSV generation

In [53]:
def copy_to_neo4j_import(node_files: dict, neo4j_import: Path):
    if not neo4j_import.exists():
        print(f"Carpeta import/ no encontrada: {neo4j_import}")
        print("Copia manualmente los CSVs desde:")
        print(f"  {next(iter(node_files.values())).parent}")
        print()
        return

    print("Copiando CSVs a Neo4j import/:")
    for rank, src in node_files.items():
        dst = neo4j_import / src.name
        shutil.copy2(src, dst)
        print(f"  {src.name} -> {dst}")
    print()

# Magic

In [54]:
if __name__ == "__main__":

    # 1. Cargar taxonomia
    df = load_taxonomy(TAXONOMY_CSV)

    # 2. Extraer nodos Kingdom a Genus
    node_files = extract_all_ranks(df, RANKS, OUTPUT_DIR)

    # 3. Extraer nodos de Species
    species_file = extract_species_nodes(df, OUTPUT_DIR)
    node_files["Species"] = species_file

    # 4. Generar Cypher
    cypher = generate_cypher_load_csv(RANKS)
    CYPHER_OUT.write_text(cypher, encoding="utf-8")
    print(f"Cypher guardado en: {CYPHER_OUT}")
    print()

    # 5. Copiar a Neo4j import/
    copy_to_neo4j_import(node_files, NEO4J_IMPORT)

Cargando gbif_taxonomy.csv ...
   4,124,085 filas cargadas
  Columnas: ['species_key', 'scientific_name', 'canonical_name', 'rank', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'taxonomic_status', 'kingdom_key', 'phylum_key', 'class_key', 'order_key', 'family_key', 'genus_key']

Extrayendo nodos por rango:
  Rango        Nodos unicos   Archivo
  ------------------------------ ---------------   ------------------------------
  Kingdom                 8   kingdom_nodes.csv
  Phylum                274   phylum_nodes.csv
  Class                 753   class_nodes.csv
  Order               2,815   order_nodes.csv
  Family             27,073   family_nodes.csv
  Genus             266,450   genus_nodes.csv

Extrayendo nodos de Species ...
  Total especies (accepted): 2,552,973
  Guardado en: species_nodes.csv

  Distribucion por reino:
    Animalia              1,810,100
    Plantae                 441,397
    Fungi                   171,665
    Chromista               